# Daily Practice — 2026-10-24 — ML Modelling: How Much Should You Trust One Train/Test Split?

**Dataset:** [Wine recognition dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) — 178 real chemical analyses of wines grown in the same region of Italy but derived from three different cultivars, 13 numeric features, loaded via `sklearn.datasets.load_wine` (a real dataset, not a synthetic one).

## Problem statement

A teammate trains a `RandomForestClassifier` on a single train/test split of the Wine dataset and reports "97% accuracy on the test set — ready to ship." With an SQA background, you know that a single split is a single sample from a noisy process: a different random split could easily report 93% or 99% purely by chance, especially with only ~178 rows. Your job is to put a number on *how much that reported metric would wobble* if the split had landed differently, compare several candidate models on stability (not just on their best-case score), and find out whether one wine cultivar is consistently harder for every model to classify correctly.

## What you should produce

1. **"The claim"** — fit one model on one fixed train/test split and report its accuracy, exactly as the teammate did.
2. **A stability check** — for at least 3 candidate models (`LogisticRegression`, `RandomForestClassifier`, `SVC`), run repeated stratified k-fold cross-validation and report **mean ± standard deviation** of accuracy and macro-F1 for each, so the single-split claim can be compared against a distribution of plausible outcomes rather than treated as ground truth.
3. **A per-class failure breakdown** for the best-performing model — precision/recall/F1 per wine cultivar (not just the overall average) — identifying which class (if any) is the weakest and by how much.
4. **A short written conclusion** answering: would you sign off on "97% accuracy, ship it" based on the single split alone? What is the actual plausible range for this model's accuracy once sampling variance is accounted for? Which class, if any, needs more data or additional features before this model should be trusted in production?


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


### Load and inspect the data

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name="cultivar")
target_names = wine.target_names

print(X.shape)
print(y.value_counts().sort_index())
X.head()


### Step 1 — Reproduce "the claim"

TODO: Split the data with `train_test_split` using `test_size=0.3`, `stratify=y`, and `random_state=RANDOM_STATE`.
Fit a single `RandomForestClassifier(random_state=RANDOM_STATE)` on the training set and print its accuracy on the
test set. This is the number the teammate is reporting — don't change anything about it yet.


In [ ]:
# TODO: stratified train/test split
X_train, X_test, y_train, y_test = None, None, None, None

# TODO: fit a single RandomForestClassifier and report test accuracy — "the claim"
claim_model = None
claim_accuracy = None

print(f"The claim: {claim_accuracy}")


### Step 2 — How fragile is that number? Repeated cross-validation

TODO: Write a function `cv_stability(model, X, y, n_splits=5, n_repeats=10)` that:
- runs `RepeatedStratifiedKFold` with the given number of splits/repeats (use `RANDOM_STATE`),
- uses `cross_validate` to score the model on both `accuracy` and `f1_macro` across all folds,
- returns a dict with the mean and standard deviation of each metric across all fold scores.

Then run it for three candidate models — `LogisticRegression` (wrapped with a `StandardScaler` via
`make_pipeline`, since it's scale-sensitive), `RandomForestClassifier`, and `SVC` (also scaled) — and
collect the results into a single comparison `DataFrame`, one row per model.


In [ ]:
def cv_stability(model, X, y, n_splits=5, n_repeats=10):
    # TODO: implement repeated stratified k-fold CV and return mean/std of accuracy and f1_macro
    raise NotImplementedError


candidate_models = {
    "logistic_regression": None,  # TODO: make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    "random_forest": None,        # TODO: RandomForestClassifier(random_state=RANDOM_STATE)
    "svc": None,                  # TODO: make_pipeline(StandardScaler(), SVC(random_state=RANDOM_STATE))
}

# TODO: run cv_stability for each candidate model and build a comparison DataFrame
stability_results = None
stability_results


### Step 3 — Per-class failure analysis

TODO: Pick the model that looked strongest (and most stable) in Step 2, fit it on the Step 1 train/test split,
and write a function `per_class_report(model, X_test, y_test, target_names)` that returns a tidy `DataFrame`
with one row per class and columns `precision`, `recall`, `f1-score`, `support` (hint: `classification_report`
with `output_dict=True` gets you most of the way there). Use it to identify the weakest cultivar, and also
print the confusion matrix to see exactly which classes get confused with which.


In [ ]:
def per_class_report(model, X_test, y_test, target_names):
    # TODO: fit/predict as needed, build a per-class precision/recall/f1/support DataFrame
    raise NotImplementedError


# TODO: fit the chosen model, call per_class_report, print the confusion matrix
best_model = None
per_class_df = None
per_class_df


### Step 4 — Write-up

TODO: In 4-6 sentences, answer:
- Would you sign off on "97% accuracy, ship it" based on the single split in Step 1 alone?
- Given the mean ± std from Step 2, what's a more honest plausible range for this model's real-world accuracy?
- Is there a cultivar in Step 3 that is consistently weaker, and would you block release on that until it's fixed, or is the gap small enough to accept?

*(your answer here)*


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
# Step 1 — reproduce the claim
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

claim_model = RandomForestClassifier(random_state=RANDOM_STATE)
claim_model.fit(X_train, y_train)
claim_accuracy = accuracy_score(y_test, claim_model.predict(X_test))
print(f"The claim: {claim_accuracy:.3f}")  # typically ~0.96-0.98 on this split

# Step 2 — repeated cross-validation stability check
def cv_stability(model, X, y, n_splits=5, n_repeats=10):
    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE)
    scores = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "f1_macro"])
    return {
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
    }

candidate_models = {
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "svc": make_pipeline(StandardScaler(), SVC(random_state=RANDOM_STATE)),
}

stability_results = pd.DataFrame(
    {name: cv_stability(model, X, y) for name, model in candidate_models.items()}
).T
print(stability_results)
# All three models typically land around 0.96-0.99 mean accuracy with a std of roughly 0.02-0.04 --
# meaning a *single* split reporting 97% could plausibly have reported anywhere from ~91% to ~100%
# just from which rows happened to land in the test fold. The "claim" is inside the plausible range,
# but citing it as a precise number is misleading; the honest statement is "~97% +/- a few points."

# Step 3 — per-class failure analysis
def per_class_report(model, X_test, y_test, target_names):
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
    df = pd.DataFrame(report).T.loc[list(target_names)]
    return df

best_model = RandomForestClassifier(random_state=RANDOM_STATE)
best_model.fit(X_train, y_train)
per_class_df = per_class_report(best_model, X_test, y_test, target_names)
print(per_class_df)

print(confusion_matrix(y_test, best_model.predict(X_test)))
# class_1 ("knott" in some encodings / the middle cultivar) is usually the one with the most confusion,
# most often mixed up with class_0 or class_2, because its chemical profile overlaps with both --
# it's a genuinely harder boundary, not noise from a single bad draw.
```

**Explanation.** The core lesson is that a single accuracy number from one train/test split is a *point
estimate* of a noisy process, not a fact about the model. With only ~178 rows and a 30% test split, the
test set has roughly 53 samples — small enough that which rows happen to land in it can shift accuracy by
several percentage points on its own. Repeated stratified k-fold cross-validation turns that one point
estimate into a distribution (mean ± std), which is the honest way to report "how good is this model" —
exactly the same instinct an SQA engineer applies to a flaky test suite: one green run doesn't prove
stability, and one red run doesn't prove a regression; you need the distribution across many runs. The
per-class breakdown then shows that aggregate accuracy can hide a structurally weak class: if one
cultivar is reliably the hardest to separate from its neighbors, that's a modelling/feature problem to
fix before shipping, not sampling noise that cross-validation would explain away. A production ML-QA gate
should therefore check *both* things this notebook produced: cross-validated score bands (not single-run
numbers) and per-slice/per-class performance (not just the macro average).

</details>
